In [9]:
import numpy as np
import pandas as pd
import zarr
import os
import matplotlib.pyplot as plt
plt.style.use('../libs/my_style.mplstyle')

import os
import sys

sys.path.append(os.path.abspath(".."))

def vec(df):
    # 1. 前処理: frameでソートしておく（groupby内での処理を減らすため）
    df = df.sort_values(['particle', 'frame'])

    # 2. 差分計算（各グループの最初の一行はNaNになる）
    # diff() は pandas のメソッドを使うとインデックスが維持されるので安全です
    dx = df.groupby('particle')['x'].diff()
    dy = df.groupby('particle')['y'].diff()

    # 3. 速度ベクトルと単位ベクトルの計算
    v = np.sqrt(dx**2 + dy**2)

    df['theta'] = dx/v + 1j * dy/v

    return df


folder = '/Volumes/My Passport/Sasaki/MTsingleBeads'

MT_path = 'MTtrack.csv'
beads_path = 'beads_tracks.csv'
exp = "20260122/exp002"

pathM = os.path.join(folder, exp, MT_path)
pathB = os.path.join(folder, exp, beads_path)

dfM = pd.read_csv(pathM)
dfM = vec(dfM)

dfB = pd.read_csv(pathB)
dfB = vec(dfB)

def calculate_polar_order(df):
    """
    各フレームごとのポーラー度（Order Parameter）を算出する
    """
    # 1. 各フレームごとに複素ベクトルの平均を取る
    # thetaには既に単位ベクトル（dx/v + i*dy/v）が入っている前提
    group_avg = df.groupby('frame')['theta'].mean()
    
    # 2. 平均ベクトルの絶対値（長さ）を計算する
    # これが 1 に近いほど揃っており、0 に近いほどバラバラ
    polar_order = group_avg.abs()

    # 各フレームごとの粒子の数をカウント
    # 'theta' が NaN でないもの（速度が計算できているもの）だけを数えるのが実用的です
    counts = df.dropna(subset=['theta']).groupby('frame')['particle'].count()
    
    return polar_order, counts


In [10]:
x = dfB['x']
y = dfB['y']
frame = dfB['frame']

In [13]:
Lum = 5
scale = 0.11
L = Lum/scale

for frame, data in dfB.groupby('frame'):
    current = dfM[dfM['frame']==frame]
    for _, i in data.groupby('particle'):
        center_x = i['x'].iloc[0]
        center_y = i['y'].iloc[0]
        local = current[(center_x-L/2<current['x']) & (current['x']<center_x+L/2) & (center_y-L/2<current['y']) & (current['y']<center_y+L/2)]['theta']
        print(local)

Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
7    0.047415+0.998875j
Name: theta, dtype: complex128
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
13    0.313503-0.949587j
Name: theta, dtype: complex128
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype: complex128)
Series([], Name: theta, dtype

In [12]:
local

Series([], Name: theta, dtype: complex128)